# Comparing deer detectors on v3

I put these ten runs together so we can see what actually improves deer detection and what it costs to run. They all use the same 3,789 images and boxes. GFL reads COCO annotations and YOLO reads the matching YOLO labels.

I’m using the fixed v3 splits: **3,193 train, 358 validation and 238 test**. Training decoys get four times the exposure because they’re closest to what we’ll see at the competition. I’ll use validation to pick weights and confidence thresholds, then score the frozen models on test.

## Results from the completed runs

I evaluated four models on the same frozen Deer v3 splits with our shared evaluator. Validation set the confidence cutoff for each model; I kept it fixed for test. Precision, recall and F1 use IoU 0.50. Count MAE is the average absolute count error per frame. The rest of the notebook preserves the original ten-run plan; the other candidates have no comparable test result.

### Validation · 358 frames, 1,436 labeled deer boxes

| Model | AP50 | AP50–95 | Precision | Recall | F1 | Confidence | Count MAE |
|---|---:|---:|---:|---:|---:|---:|---:|
| YOLO11s | 70.9% | 42.7% | 93.2% | 67.4% | 78.2% | 0.40 | 1.29 |
| YOLO26s | 73.3% | 43.9% | 95.0% | 68.1% | 79.3% | 0.31 | 1.25 |
| GFL | 76.1% | 44.6% | 92.6% | 70.4% | 80.0% | 0.51 | 1.12 |
| D-FINE-L | 81.7% | 58.6% | 91.8% | 76.7% | 83.5% | 0.45 | 1.00 |

### Held-out test · 238 frames, 575 labeled deer boxes

| Model | AP50 | AP50–95 | Precision | Recall | F1 | Confidence | Count MAE |
|---|---:|---:|---:|---:|---:|---:|---:|
| YOLO11s | 60.7% | 29.5% | 80.3% | 43.8% | 56.7% | 0.40 | 1.24 |
| YOLO26s | 70.3% | 33.6% | 71.2% | 66.3% | 68.6% | 0.31 | 0.87 |
| GFL | 69.5% | 31.4% | 71.8% | 67.3% | 69.5% | 0.51 | 0.82 |
| D-FINE-L | 65.6% | 33.1% | 42.4% | 74.6% | 54.1% | 0.45 | 2.11 |

YOLO26s had the highest test AP. GFL was slightly ahead on F1 and count error. D-FINE-L matched 429 boxes versus 381 for YOLO26s, but left 583 unmatched predictions versus 154. Those extra boxes are a lot for an operator to review. I’d start flight trials with YOLO26s. The test set is mixed-source and these are frame-level counts; Jetson performance and unique deer counted per flight still need to be measured.

### YOLO26s on dry grass and decoys

The same saved YOLO26s weights were used for both passes below. At confidence 0.31, four overlapping crops recovered more deer on the BAMBI and Mississippi test subset, with more unmatched boxes and about four times the inference time on my Mac.

| Mode | BAMBI matches | Mississippi decoy matches | Unmatched boxes | Median inference |
|---|---:|---:|---:|---:|
| Full frame | 24 / 73 | 11 / 14 | 10 | 72 ms |
| Four overlapping crops | 34 / 73 | 12 / 14 | 26 | 286 ms |

That crop pilot used test footage we’d already inspected, so I’m treating it as a diagnostic. Next I’d collect labeled dry-field flights and verified empty passes, hold out whole flights, and compare a short YOLO26s fine-tune to these unchanged weights. I’d try crops as a second pass and measure how many distinct deer we recover for the extra review time.

The [experiment record](../../docs/experiments/summaries/2026-11-10-deer-v3-modal-comparison.md) has checkpoint paths and hashes, plot and prediction locations, and the detailed error audit. No cells need to run to read these results.

## Set up the data

This notebook preserves the original workstation plan. For the completed Modal results and checkpoint locations, see [the experiment record](../../docs/experiments/summaries/2026-11-10-deer-v3-modal-comparison.md).

Place the shared ZIP in the workspace input folder or set DEER_V3_DRIVE_ID; see the prototype README.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt

folder = Path.cwd()
sys.path.insert(0, str(folder.parents[1]))
from dataloaders.deer_shared.common import read_json, write_json
from workflow import candidate_stage, prepare, plot_candidate_training, resolution_rows, summarize

workspace = Path(os.environ.get('DEER_COMPARE_WORKSPACE', Path.home() / 'deer_v3_compare'))
output = workspace / 'runs/comparison_v1'
data = prepare(workspace)
gfl_workspace = Path.home() / 'gfl_workspace'
gfl_python = gfl_workspace / '.venv-gfl/bin/python'
yolo_python = Path.home() / 'code/.venv/bin/python'
run_gpu = os.environ.get('DEER_COMPARE_RUN') == '1'
print('GPU stages enabled.' if run_gpu else 'Preview mode. GPU stages are paused.')


## What went into v3

I picked imagery for the downward camera we expect to fly over an open field in Medicine Hat. The decoys are our closest match. I tightened oversized boxes and kept frames with coyote or bird decoys as negatives when the original key confirmed there were no deer. Young Deer gives us good aerial views, mostly in green grass. BAMBI adds some brown open ground, AWIR adds RGB pen views, and Sherbrooke adds crowded summer scenes.

I reviewed 1,029 Rucervus candidates and kept 454 aerial views, including dry vegetation and greener field edges. We used image matching and capture information to keep related views together where we could. BAMBI is grouped by location, AWIR by date, and Young Deer and Rucervus use inferred scene groups.

| Source | Train | Validation | Test |
|---|---:|---:|---:|
| Young Deer | 1,570 | 48 | 60 |
| BAMBI | 831 | 44 | 56 |
| Mississippi decoys | 139 | 31 | 28 |
| AWIR RGB | 34 | 6 | 13 |
| Sherbrooke summer | 328 | 147 | 0 |
| Rucervus aerial | 291 | 82 | 81 |
| **Total images** | **3,193** | **358** | **238** |
| Deer boxes | 9,848 | 1,436 | 575 |
| Images with no deer boxes | 99 | 103 | 20 |

We still have limited dry yellow grass at competition-like angles, and our decoys come from one field. I’ll look at decoy results separately because good scores on live deer can hide misses on our actual targets.

## What I’m comparing

Anant’s original decoy baseline used **YOLO11s**. I’m retraining the same YOLO11s architecture on v3 as Anant’s baseline. YOLO26s is the additional candidate I’m testing. **GFL 1 and GFL 2 are my runs.** They use the same detector with different training schedules.

GFL and YOLO both detect boxes directly in one pass. My **GFL** model uses a ResNeXt-101 backbone with deformable convolutions and a feature pyramid. Its head learns a score that combines class confidence and box quality, and predicts a distribution for each box edge. The deformable layers can shift where they sample image features. [GFL paper](https://arxiv.org/abs/2006.04388)

**YOLO11s** predicts boxes across multiple feature scales and uses non-maximum suppression to remove duplicate detections. **YOLO26s** supports an end-to-end head that produces detections without that suppression step. I want to see how those differences affect misses, duplicate boxes and inference cost on our data. [YOLO11](https://docs.ultralytics.com/models/yolo11) · [YOLO26](https://docs.ultralytics.com/models/yolo26)

## 1080p camera, different model input sizes

The drone sends **1920 × 1080** video. The receiver decodes each frame at that resolution and the detector resizes its own copy. Model input size is separate from the camera and stream setting. With aspect ratio preserved, a 1920 × 1080 frame becomes about 960 × 540 at 960 or 1280 × 720 at 1280 before padding. A 30-pixel-wide deer becomes roughly 15 pixels wide at 960 and 20 at 1280. The DETR recipes in this notebook resize to a square, so their vertical scale differs.

I’ll keep the stream at 1080p. Eight runs use 1280. I’ll also train YOLO26s at 960 and 640 to see what smaller inputs save in time and cost in recall. All three YOLO26s checkpoints will see the same validation frames at all three sizes. [Ultralytics resize and padding](https://docs.ultralytics.com/modes/predict/)

| Run | Training input | Parameters | FP32 weights (MiB) | Where I’ll try it first |
|---|---:|---:|---:|---|
| YOLO11s — Anant’s architecture | 1280 | 9,428,179 | 36.0 | Jetson |
| YOLO26s | 1280 | 9,948,638 | 38.0 | Jetson |
| YOLO26s — resolution trial | 960 | 9,948,638 | 38.0 | Jetson |
| YOLO26s — low-resolution trial | 640 | 9,948,638 | 38.0 | Jetson speed check |
| GFL 1 — short | 1280 | 53,284,958 | 203.3 | Ground laptop over RTSP |
| GFL 2 — long | 1280 | 53,284,958 | 203.3 | Ground laptop over RTSP |
| RTMDet-tiny | 1280 | 4,873,179 | 18.6 | Jetson |
| RT-DETRv2-R18 | 1280 | 20,083,028 | 76.6 | Jetson |
| D-FINE-L | 1280 | 31,081,807 | 118.6 | Ground laptop over RTSP |
| RT-DETRv2-L (R50) | 1280 | 42,729,027 | 163.0 | Ground laptop over RTSP |

I counted the parameters in each one-class model before inference optimizations. FP32 weights is four bytes per parameter. The final table will show the saved checkpoint size, which can include optimizer state and other data. Peak GPU memory is measured separately.

GFL on Jetson is a separate check. I’ll try the existing checkpoint once we have the board.

## How I’ll run the comparison

I’ll use the same frozen data for all ten runs. The separate 1280 FP16 pass uses the same validation frames for every checkpoint. The frameworks keep their own resize, padding and augmentation. The YOLO runs share their training recipe apart from the 960 and 640 input size trials. Both GFL runs start from the same COCO weights, with different epoch budgets and LR schedules.

I’ll run one job at a time on the 3070, with a small training smoke check first. Each section picks weights on validation AP, picks a confidence threshold on validation F1, then scores test. I’ll freeze the ten recipes before the first test result and keep test out of any later tuning.

I’ll finish each model’s training and evaluation before starting the next one. If a stage fails, the notebook stops and keeps the checkpoints and logs so I can recover that model.

In [ ]:
def show_image(path):
    if path.is_file():
        display(Image(filename=str(path)))

def run_stage(model, stage):
    run = output / model
    only = os.environ.get('DEER_COMPARE_ONLY')
    if run_gpu and (not only or only == model):
        status_path = output / 'status.json'
        status = read_json(status_path) if status_path.exists() else {}
        try:
            if stage != 'train' and not (run / 'training_complete.json').exists():
                raise RuntimeError('Waiting for completed training.')
            candidate_stage(stage, model, workspace, data, output, gfl_python, yolo_python, gfl_workspace)
        except Exception as exc:
            status.setdefault(model, {})[stage] = str(exc)
            write_json(status_path, status)
            raise
        status.setdefault(model, {})[stage] = 'complete'
        write_json(status_path, status)
        print(f'{model} {stage}: complete')
    else:
        queue_file = output / 'unattended.json'
        queue = read_json(queue_file) if queue_file.exists() else {}
        entry = queue.get(model, {})
        if entry.get('state') == 'skipped' and entry.get('stage') == stage:
            print(f"{model} {stage}: skipped ({entry['reason']})")
        elif (run / ('training_complete.json' if stage == 'train' else f'{stage}/metrics.json')).is_file() or (
                stage == 'profile' and any(run.glob('profile_*/profile.json'))):
            print(f'{model} {stage}: saved results')
        else:
            print(f'{model} {stage}: waiting for the run')
    return run

def show_scores(run, split):
    file = run / split / 'metrics.json'
    if not file.is_file():
        return
    metrics = read_json(file)
    lines = ['| Source | Images | AP50 | AP50–95 | Precision | Recall | F1 | Count MAE | Exact count |',
             '|---|---:|---:|---:|---:|---:|---:|---:|---:|']
    for source, row in [('All', metrics), *metrics['by_source'].items()]:
        numbers = [row[k] for k in ('ap50', 'ap50_95', 'precision', 'recall', 'f1', 'count_mae', 'exact_count_accuracy')]
        values = ' | '.join('—' if x is None else f'{x:.3f}' for x in numbers)
        lines.append(f"| {source} | {row['images']} | {values} |")
    display(Markdown(chr(10).join(lines)))
    print(f"Threshold {metrics['threshold']:.2f} from validation. "
          f"False positives on {metrics['negative_images_with_false_positives']}/{metrics['negative_images']} empty frames.")

def show_profile(run):
    lines = ['| Input | Images | Median ms | p95 ms | Peak CUDA allocated MiB | Peak CUDA reserved MiB | Peak host RSS MiB | Sample AP50–95 | Sample recall |',
             '|---|---:|---:|---:|---:|---:|---:|---:|---:|']
    for setting in ('test', 'profile_1280_fp32', 'profile_960_fp32', 'profile_640_fp32', 'profile_640_fp16', 'profile_960_fp16', 'profile_1280_fp16'):
        split = 'test' if setting == 'test' else 'val sample'
        file = run / setting / 'profile.json'
        if file.exists():
            row = read_json(file)
            label = f"{row['requested_size']} {row['precision'].upper()} · {split}"
            sample_file = run / setting / 'sample_metrics.json'
            sample = read_json(sample_file) if sample_file.is_file() else {}
            ap, recall = ('—' if sample.get(k) is None else f'{sample[k]:.3f}'
                          for k in ('ap50_95', 'recall'))
            lines.append(f"| {label} | {row['images']} | {row['median_ms']:.1f} | {row['p95_ms']:.1f} | "
                         f"{row['peak_allocated_mib']:.0f} | {row['peak_reserved_mib']:.0f} | "
                         f"{row['peak_host_rss_mib']:.0f} | {ap} | {recall} |")
    if len(lines) > 2:
        display(Markdown(chr(10).join(lines)))

## YOLO11s — Jetson candidate, follow-up to Anant’s baseline

Anant’s original run used YOLO11s. I’m retraining that architecture on v3 from official COCO detection weights. The run uses 1280 input, up to 100 epochs and patience 30.

### Training

The per-epoch precision and recall curves come from the Ultralytics validator.

In [ ]:
run = run_stage('yolo11s', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    native = run / 'fit/results.png'
    show_image(native if native.is_file() else run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('yolo11s', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('yolo11s', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

In [ ]:
run = run_stage('yolo11s', 'profile')
show_profile(run)

## YOLO26s — my Jetson candidate

I’m using the same training settings as YOLO11s, including up to 100 epochs and patience 30. This run tests the newer detector and its end-to-end head.

### Training

The per-epoch precision and recall curves come from the Ultralytics validator.

In [ ]:
run = run_stage('yolo26s', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    native = run / 'fit/results.png'
    show_image(native if native.is_file() else run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('yolo26s', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('yolo26s', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

I’ll compare AP and recall at 640, 960 and 1280 on the same 48 validation frames. Recall uses this checkpoint’s fixed validation threshold. I’ll repeat the timing check on Jetson with the real camera stream.

In [ ]:
run = run_stage('yolo26s', 'profile')
show_profile(run)

## YOLO26s at 960 — my 960 Jetson trial

I’m training a second YOLO26s checkpoint at 960. It starts from the same pretrained weights and uses the same data and settings as the 1280 run. At 960, a 1080p frame becomes about 960 × 540 before padding. I want to see how much speed we gain at 960 and how much recall we give up compared with 1280.

### Training

The same 100-epoch cap and patience 30 apply. The input size is the one training setting I’m changing.

In [ ]:
run = run_stage('yolo26s_960', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    native = run / 'fit/results.png'
    show_image(native if native.is_file() else run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('yolo26s_960', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('yolo26s_960', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

I’ll compare AP and recall at 640, 960 and 1280 on the same 48 validation frames. Recall uses this checkpoint’s fixed validation threshold. I’ll repeat the timing check on Jetson with the real camera stream.

In [ ]:
run = run_stage('yolo26s_960', 'profile')
show_profile(run)

## YOLO26s at 640 — my low-resolution trial

I’m training a third YOLO26s checkpoint at 640 with the same starting weights, data and training settings. At 640, a 1080p frame becomes about 640 × 360 before padding. I want to see whether the speed gain is worth the loss in small-deer recall.

### Training

The same 100-epoch cap and patience 30 apply.

In [ ]:
run = run_stage('yolo26s_640', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    native = run / 'fit/results.png'
    show_image(native if native.is_file() else run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('yolo26s_640', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('yolo26s_640', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

I’ll compare AP and recall at 640, 960 and 1280 on the same 48 validation frames. Recall uses this checkpoint’s fixed validation threshold. I’ll repeat the timing check on Jetson with the real camera stream.

In [ ]:
run = run_stage('yolo26s_640', 'profile')
show_profile(run)

## GFL 1 — my short run, ground first

I’m using ResNeXt-101 with DCNv1, COCO initialization and a 10-epoch schedule. The learning rate drops at epochs 6 and 8.

### Training

I reuse the epoch validation predictions for precision and recall at confidence 0.25 and IoU 0.5. This adds no second validation pass.

In [ ]:
run = run_stage('gfl_short', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    show_image(run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('gfl_short', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('gfl_short', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

In [ ]:
run = run_stage('gfl_short', 'profile')
show_profile(run)

## GFL 2 — my longer run, ground first

I’m starting again from the same COCO weights with up to 100 epochs and patience 30. The learning rate drops at epochs 60 and 80 if the run reaches them.

### Training

I reuse the epoch validation predictions for precision and recall at confidence 0.25 and IoU 0.5. This adds no second validation pass.

In [ ]:
run = run_stage('gfl_long', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    show_image(run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('gfl_long', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('gfl_long', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

In [ ]:
run = run_stage('gfl_long', 'profile')
show_profile(run)

## RTMDet-tiny — my Jetson candidate

I’m fine-tuning a COCO-pretrained CSPNeXt detector for 60 epochs at 1280. It uses a lightweight convolutional backbone and a dense detection head with NMS. I’ll keep the aspect ratio when resizing.

### Training

I’ll plot loss and native validation AP across epochs. Precision and recall are in the shared validation and test reports.

In [ ]:
run = run_stage('rtmdet_tiny', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    show_image(run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('rtmdet_tiny', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('rtmdet_tiny', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

In [ ]:
run = run_stage('rtmdet_tiny', 'profile')
show_profile(run)

## RT-DETRv2-R18 — my Jetson transformer candidate

I’m fine-tuning the official COCO-pretrained R18 checkpoint for 60 epochs at 1280. Its decoder uses object queries to return a set of detections without NMS. I’ll use the upstream square resize recipe.

### Training

I’ll plot loss and native validation AP across epochs. Precision and recall are in the shared validation and test reports.

In [ ]:
run = run_stage('rtdetrv2_r18', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    show_image(run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('rtdetrv2_r18', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('rtdetrv2_r18', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

In [ ]:
run = run_stage('rtdetrv2_r18', 'profile')
show_profile(run)

## D-FINE-L — my ground-laptop candidate

I’m fine-tuning the COCO-pretrained D-FINE-L for 60 epochs at 1280 × 1280. It refines box edges across decoder layers. I want to see if it finds more small deer on streamed footage.

### Training

I’ll plot loss and validation AP by epoch. Precision and recall are in the validation and test reports.

In [ ]:
run = run_stage('dfine_l', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    show_image(run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('dfine_l', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('dfine_l', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

In [ ]:
run = run_stage('dfine_l', 'profile')
show_profile(run)

## RT-DETRv2-L (R50) — my ground-laptop candidate

I’m fine-tuning the larger ResNet-50 version for 60 epochs at 1280 × 1280. I’ll compare it with the R18 model and report their input sizes alongside the scores.

### Training

I’ll plot loss and validation AP by epoch. Precision and recall are in the validation and test reports.

In [ ]:
run = run_stage('rtdetrv2_r50', 'train')
if (run / 'training_complete.json').exists():
    fit = read_json(run / 'training_complete.json')
    print(f"Training wall time: {fit['seconds'] / 3600:.2f} h")
    print(f"Selected checkpoint: {fit['best']}")
    plot_candidate_training(run)
    show_image(run / 'training_curves.png')

### Validation

I’ll pick the confidence threshold here and keep it fixed for test.

In [ ]:
run = run_stage('rtdetrv2_r50', 'val')
show_scores(run, 'val')
show_image(run / 'val/threshold_curves.png')

### Held-out test

I’ll score the frozen checkpoint and threshold, then look at misses, false positives and count errors.

In [ ]:
run = run_stage('rtdetrv2_r50', 'test')
show_scores(run, 'test')
show_image(run / 'test/count_errors.png')
show_image(run / 'test/error_gallery.png')

### Inference cost

Batch one with file decode, preprocessing, inference and postprocessing. The main FP16 row uses the test split. The other rows use the same fixed validation sample.

In [ ]:
run = run_stage('rtdetrv2_r50', 'profile')
show_profile(run)

## Compare all ten runs

I’ll compare test AP and count error alongside latency and memory, with a separate look at the 28 decoy test images. Native epoch metrics and loss scales differ by framework. The final test table uses the same evaluator and matching rules for all ten.

The main test table includes each run’s input size. That comparison covers the whole recipe, including resize. The separate 1280 pass compares inference cost on the same validation frames.

I’m putting decoy recall, misses and extra boxes next to pooled AP and latency. This is how I’ll shortlist the flight model. I’ll check that shortlist on the Jetson or ground laptop with the camera stream running before choosing what to fly.

For the Saturday flight test, I’ll pick the best Jetson candidate that keeps up with the camera. I’ll make that pick even if a ground-laptop model scores higher overall.

In [ ]:
rows = summarize(output)
tests = [row for row in rows if row['split'] == 'test']
lines = ['| Run | Input | Test AP50–95 | Test F1 | Decoy recall | Decoy misses | Extra decoy boxes | Decoy count MAE | p95 ms | Peak CUDA MiB | Inference params (M) | Checkpoint MiB |',
         '|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for row in tests:
    recall = row['decoy_recall']
    mae = row['decoy_count_mae']
    decoy_recall = '—' if recall is None else f'{recall:.3f}'
    decoy_mae = '—' if mae is None else f'{mae:.3f}'
    lines.append(f"| {row['model']} | {row['requested_size']} | {row['ap50_95']:.3f} | {row['f1']:.3f} | "
                 f"{decoy_recall} | {row['decoy_fn']} | {row['decoy_fp']} | {decoy_mae} | "
                 f"{row['p95_ms']:.1f} | {row['peak_reserved_mib']:.0f} | "
                 f"{row['parameters']/1e6:.2f} | {row['checkpoint_bytes']/2**20:.1f} |")
if tests:
    display(Markdown(chr(10).join(lines)))
    print(f'{len(tests)} of 10 runs have test results. CUDA memory is peak reserved on the 3070. Checkpoint size is the saved file on disk.')
for name in ('learning_curves.png', 'comparison.png'):
    show_image(output / name)

resolution = resolution_rows(output)
if resolution:
    lines = ['### YOLO26s resolution trial', '',
             'Same validation frames and batch-one FP16 across all rows.',
             '',
             '| Train size | Inference size | Sample AP50–95 | Recall | Count MAE | p95 ms | Peak CUDA MiB |',
             '|---:|---:|---:|---:|---:|---:|---:|']
    for row in resolution:
        ap = '—' if row['ap50_95'] is None else f"{row['ap50_95']:.3f}"
        lines.append(f"| {row['training_size']} | {row['inference_size']} | {ap} | {row['recall']:.3f} | "
                     f"{row['count_mae']:.3f} | {row['p95_ms']:.1f} | {row['peak_reserved_mib']:.0f} |")
    display(Markdown(chr(10).join(lines)))
print(f'Checkpoints, logs, predictions, figures and CSVs: {output}')

## Flight setup and ground laptop

**On Jetson.** I’ll start with YOLO11s or YOLO26s, build the FP16 TensorRT engine on the target board, and check that its detections match the PyTorch version. RTMDet-tiny and RT-DETRv2-R18 are the other onboard candidates. I’ll check each one with the camera running and record shared RAM, p95 latency and dropped frames. [Ultralytics Jetson guide](https://docs.ultralytics.com/guides/nvidia-jetson) · [RTMDet export](https://github.com/open-mmlab/mmdetection/tree/main/configs/rtmdet) · [RT-DETRv2](https://github.com/lyuwenyu/RT-DETR/tree/main/rtdetrv2_pytorch)

**On the ground laptop.** The Jetson sends the camera stream through our RTSP node. The laptop decodes it and runs GFL, D-FINE-L or RT-DETRv2-L on its GPU. I’ll keep a short live queue, preserve frame IDs and timestamps, and save footage onboard in case the link drops. I’ll measure the time from capture to detection and check whether compression makes small deer harder to spot. If the Jetson is an Orin Nano, budget for software H.264 encoding. [NVIDIA Orin Nano encoding guide](https://docs.nvidia.com/jetson/archives/r36.4/DeveloperGuide/SD/Multimedia/SoftwareEncodeInOrinNano.html)

**GFL on Jetson is an experiment.** I’ll try our saved checkpoint in PyTorch when we have the board. Our config uses DCNv1, and the documented MMDeploy TensorRT plugin is for DCNv2. I’ll use ground inference if the Jetson build or speed is a problem. [PyTorch for Jetson](https://docs.nvidia.com/deeplearning/frameworks/install-pytorch-jetson-platform/) · [MMDeploy TensorRT operators](https://github.com/open-mmlab/mmdeploy/blob/main/docs/en/06-custom-ops/tensorrt.md)

The authors report 54.0 COCO AP for D-FINE-L and 53.4 for RT-DETRv2-L at 640. Those are separate benchmarks. Our decision will come from deer recall, count errors and latency on our own data and stream. [D-FINE model zoo](https://github.com/Peterande/D-FINE#model-zoo) · [RT-DETR model zoo](https://github.com/lyuwenyu/RT-DETR#readme)